# BEFORE YOU RUN ANYTHING — SETUP CHECKLIST

Everything on this list happens **outside** the notebook, in the Kaggle UI. Do all five,
in order, *before* you execute a single cell. Nothing below this cell will work otherwise.

---

### 1. Accelerator → **GPU T4 x2**

Right sidebar → **Session options** → **Accelerator** → select **`GPU T4 x2`**.

Not `None`, not `GPU P100`, not `TPU`. The generator does not fit on one card and needs the
**~29 GiB combined** across both T4s, with Ollama splitting layers between them. P100 is a
single 16 GB card and will OOM.

**Measured GGUF sizes** (from the Hugging Face repo, not estimates):

| quant | weights | headroom on T4x2 (~29.2 GiB) |
|---|---|---|
| `Q4_K_M` | 24.5 GB | ~4.7 GiB — **the only quant that fits**, tight |
| `Q6_K` | 33.5 GB | none — will not load |
| `Q8_0` | 33.6 GB | none — will not load |

`Q4_K_M` is a ~24.5 GB download — the repo has no smaller quant (next is Q6_K at
33.5 GB, which does not fit). Nemotron-3.5-Lightning is a hybrid Mamba-2 + MoE
model (`nemotron_h_moe`), not a plain transformer — confirm the VRAM cell
(section 3b) loads it without error before committing a long session; at 24.5 GB
on ~29 GiB of T4x2 the KV-cache headroom is thin.

### 2. Internet → **ON**

Right sidebar → **Session options** → **Internet** → toggle **on**.

Required for: installing Ollama (`curl https://ollama.com/install.sh`), pulling the model
GGUFs (~24.5 GB generator, ~4.7 GB judge), `pip install sentence-transformers`, downloading the
MiniLM embedding model (~80 MB), and cloning the repo if you use `REPO_URL`.

Kaggle requires a **phone-verified account** to enable Internet. If the toggle is greyed
out, verify your account first — this is the single most common thing that blocks a
session before it starts.

### 3. Attach the project data — **Kaggle Dataset** *(no GitHub repo exists yet)*

There is **no GitHub remote for this project**, so use the Dataset route. The notebook
will still try a `git clone` first if you fill in `REPO_URL`, but with `REPO_URL = ""`
(the default) it goes straight to the dataset mount.

**How to create and attach the dataset:**

1. Zip the **`okf-rag-v2/` folder**, keeping its internal paths intact. It must contain:

   ```
   config.yaml                                      # v2 config extends this
   src/                                             # the whole package (9 .py files)
   questions/questions.csv                          # 100 human-authored questions
   data/raw/                                        # 40 Kubernetes docs — the CHUNK arm reads these
   results/v2_active_qids.txt                       # the FINAL 63 qids this run is scoped to
   results/v2_excluded_qids.txt                     # 7 audit exclusions   (cell 8 requires it)
   results/v2_backfill_qids.txt                     # 7 multi-hop backfill (cell 8 requires it)
   results/v2_qids.txt                              # v1's 60; cell 21 rebuilds 63 = 60 − excluded + backfill
   results/v1_reference/scored_okf_rag_v1_full.csv  # cross-check for the qid list
   ```

   **Do NOT include `data/okf_bundle/`.** v2 builds its own bundle from `data/raw/` (cell
   "4b — BUILD THE BUNDLE"); a pre-shipped bundle is v1's, and cell 8 hard-fails on purpose
   if it finds one. That folder is not in the current project tree, so a fresh zip won't
   contain one unless you add it back — don't.

   `data/raw/` is **not optional**: without it `ChunkRAG` raises `No chunks built` and the
   baseline arm cannot run at all. `scripts/` and the other `v1_reference/` files are not
   needed by the notebook, but including the whole folder (minus `data/okf_bundle/`) is
   simpler than pruning it.

2. Go to **kaggle.com/datasets** → **New Dataset**.
3. Drag the zip in. Kaggle unzips it automatically and **preserves the folder structure**.
4. Title it `okf-rag-v2`. Set visibility to **Private**. → **Create**.
5. Back in this notebook: right sidebar → **Input** → **+ Add Input** → **Datasets** tab →
   **Your Work** → select `okf-rag-v2` → **Add**.
6. Confirm the mount path shown in the Input panel. It is normally
   `/kaggle/input/okf-rag-v2`. **If it differs, edit `KAGGLE_INPUT` in the settings cell
   below to match it exactly.**

*(If you later push this project to GitHub, set `REPO_URL` in the settings cell instead and
you can skip the dataset entirely.)*

### 4. Check your remaining weekly GPU quota

Kaggle shows it top-right of the notebook editor, and under **Settings → Accelerator**.
Quota is **30 GPU-hours per week**, and one session is capped at **12 hours**.

**Expected budget for this run — 252 LLM calls total:**

| phase | calls | estimate |
|---|---|---|
| Ollama install + model pulls (~24 GB total) | — | 20–35 min |
| Smoke test (3 qids × 2 arms, gen + judge) | 12 | 5–10 min |
| Generation, chunk arm (63 qids) | 63 | 30–50 min |
| Generation, OKF arm (63 qids) | 63 | 30–50 min |
| Judging, all 126 rows | 126 | 45–75 min |
| **total** | **252** | **≈ 2–3.5 h** |

These are **estimates, not measured numbers** — nothing in this project has run on a T4
yet. They assume ~15–25 tok/s for a 30B-A3B MoE at Q4 on Turing. Budget **4 GPU-hours** to
be safe. Well inside the 12 h session cap, but do not start with under ~5 h of weekly
quota left.

### 5. Run the SMOKE TEST first — do **not** run all cells top-to-bottom

The notebook is deliberately **not** a run-all. Execute cells in order up to and including
the **SMOKE TEST** section, then **stop and actually read its output**: 6 answers and 6
judge labels, printed in full. Check they are on-topic, cite sources, and that the judge
returns a real label rather than an empty string or a wall of reasoning.

Only once that looks right do you run the full 63-question continuation cells. Every work
cell is checkpointed and re-runnable, so a session that dies mid-run resumes where it
stopped — but a *silently broken* run wastes the whole session, and the smoke test is what
catches that in 10 minutes instead of 3 hours.

---

> **The GLM-4.6 judge blocker is now resolved by an explicit user decision** — see the next
> cell. `JUDGE_MODEL` is `qwen2.5:7b-instruct`. Report that as the v2 judge, **not** GLM-4.6.


# OKF-RAG v2 — Kaggle T4x2 run (generation + judging)

**v1 remains the frozen primary result** (`results/v1_reference/`). This notebook produces a
*secondary* v2 run that changes **only the generator and judge model**. Corpus, OKF
bundle, retrieval settings, context budget, prompts and the judging rubric are all
inherited unchanged from `config.yaml`, so any difference is attributable to the model.

## Scope
- The **active 63 qids** (`results/v2_active_qids.txt` = v1's human-labelled 60, − 7 audit exclusions, + 7 multi-hop backfill)
- **Both arms** (chunk baseline + OKF) → 126 generation calls
- Then 126 judge calls → **252 LLM calls total**

## Model choice rationale
Picked off the Vectara hallucination leaderboard for **low measured hallucination rate**
and, critically, **different model families for generator vs judge** — v1's biggest
caveat was that `llama3.1:8b` judged its own generations (self-preference bias).

| role | model | leaderboard hallucination rate |
|---|---|---|
| generation | `nvidia/Nemotron-3.5-Lightning-30B-A3B` | — |
| judge | `zai-org/GLM-4.6` | 9.5% |

*Nemotron-3.5-Lightning is the model originally tested via the NVIDIA API for this project; it corrects an earlier incorrect model string.*

Both run **locally inside this notebook via Ollama** — no external API, so none of the
Groq / Gemini / OpenRouter / NVIDIA quota walls this project has already hit apply.

### Why Ollama and not vLLM
Kaggle's T4 is Turing: no bf16, no FP8, and vLLM's quantised-MoE kernels are unreliable
there. Ollama installs with one curl, ships GGUF quants that fit, unloads one model
before loading the next, and already serves `/v1/chat/completions` — the exact dialect
`src/llm.py` speaks. Zero pipeline code changes.

## ⚠️ VRAM reality check — read before you burn a session

### Judge: GLM-4.6 was requested, and is impossible here

**`zai-org/GLM-4.6` is a ~355B-parameter MoE. It will not run on T4x2 (~29 GiB).** Even at
4-bit its weights are roughly 200 GB. There is no quantisation that fits.

**The user's decision (2026-08-28): the v2 judge is `qwen2.5:7b-instruct`.** This is a
substitution and it is recorded as one — it is *not* GLM-4.6 and must never be reported as
GLM-4.6. Rationale: pulled straight from Ollama's own library (no GGUF repo hunting),
~4.7 GB, and a **different model family from the Nemotron generator**, which is the property
that actually retires v1's self-preference-bias caveat.

| candidate | size @ Q4 | fits? | note |
|---|---|---|---|
| `hf.co/zai-org/GLM-4.6-GGUF:Q4_K_M` | ~200 GB | **no** | originally requested, impossible on this hardware |
| **`qwen2.5:7b-instruct`** | **~4.7 GB** | **yes** | **chosen** — different family, Ollama library |
| `qwen3:30b-a3b` | ~18 GB | yes | different family, but competes with the generator for VRAM |
| `gemma3:27b` | ~17 GB | yes | different family, same competition problem |

### Generator quants — `NVIDIA-Nemotron-3.5-Lightning-30B-A3B-GGUF`

Repo sizes (from the Hugging Face repo, not estimates):

| quant | weights | note |
|---|---|---|
| `Q4_K_M` | **24.5 GB** | **the default** — the only quant that fits T4x2, tight |
| `Q6_K` | 33.5 GB | does not fit |
| `Q8_0` | 33.6 GB | does not fit |

`Q4_K_M` is the only usable quant here — the repo has nothing smaller. Nemotron-3.5-
Lightning is a hybrid Mamba-2 + MoE model (`nemotron_h_moe`), not a plain transformer
— confirm the VRAM cell (section 3b) loads it without error before a long session; at
24.5 GB the KV-cache headroom on T4x2 is thin and an OOM at load is possible.

### The two models never share VRAM

`OLLAMA_MAX_LOADED_MODELS=1`, and the VRAM cell calls `ollama stop` between loads. The
binding constraint is therefore **the generator alone plus its KV cache**, not generator +
judge summed. The 4.7 GB judge does not compete for space.

**Run the VRAM cell (section 3b) and read its measured numbers before committing to a full
session.** A pull that succeeded can still fail to load.

## Kaggle session budget
- **30 GPU-hours/week**, **12 h max per session**. `/kaggle/working` is the only thing
  that survives; `~/.ollama` does not, so the model pulls repeat every session
  (~24 GB, 20–35 min).
- **Run the SMOKE TEST section in a short separate session first.** Eyeball the 6 answers
  and 6 judge labels before spending a long session on the other 60 qids.
- Every work cell below is checkpointed. If the 12 h limit kills the session, reopen and
  **run all cells again** — completed `(arm, qid)` pairs are skipped, not repaid.
- **Internet must be ON** in notebook settings. Accelerator: **GPU T4 x2**.


## Where the project files come from

Covered in full in the setup checklist above (item 3). Short version:

- `REPO_URL` set  → the notebook `git clone`s it.
- `REPO_URL = ""` → it copies from `KAGGLE_INPUT` (the attached Kaggle Dataset mount).

Whichever path runs, the cell **prints which one it used** and then hard-fails naming any
missing file, rather than letting you discover it deep inside a retriever 20 minutes later.


In [ ]:
# ---------------------------------------------------------------- settings
# Everything you might want to change lives in this one cell.

REPO_URL     = ""                           # e.g. "https://github.com/you/okf-rag.git"; "" = use the Kaggle dataset
KAGGLE_INPUT = "/kaggle/input/datasets/dakshmaher22/okf-rag-v2/okf-rag-v2"  # observed mount, 2026-08-27
                                            # if wrong, the next cell auto-resolves it by searching
PROJECT      = "/kaggle/working/okf-rag"    # inside /kaggle/working so results persist

# The "hf.co/" prefix is BROKEN in current Ollama: it resolves to a huggingface.co realm and
# then rejects it -- `pull model manifest: realm host "huggingface.co" does not match
# original host "hf.co"`. Use the full "huggingface.co/" host. Verified working 2026-08-27.
GEN_QUANT = "Q4_K_M"    # 24.5 GB -- the ONLY quant that fits T4x2. Repo also has Q6_K (33.5 GB)
                        # and Q8_0 (33.6 GB); neither fits. No smaller quant exists. Headroom is thin.
GEN_MODEL   = f"huggingface.co/lmstudio-community/NVIDIA-Nemotron-3.5-Lightning-30B-A3B-GGUF:{GEN_QUANT}"
            # NB: hybrid Mamba-2 + MoE arch (nemotron_h_moe), not a plain transformer --
            # confirm Ollama loads it in the VRAM cell before trusting a long run.
JUDGE_MODEL = "qwen2.5:7b-instruct"   # ~4.7 GB, Ollama library, different family from Nemotron.
                                      # SUBSTITUTE for the unrunnable GLM-4.6 -- report THIS as the v2 judge.

# The bundle is REBUILT for v2 (decision 2026-08-28). v1's bundle was extracted by
# llama3.1:8b; reusing it made the whole v2 run a v1 result. Same model as generation --
# one model family for the whole v2 pipeline, and it is already pulled.
BUNDLE_MODEL = GEN_MODEL

# SHA-256 over v1's sorted bundle tree. The verification gate asserts the freshly built
# bundle does NOT hash to this. Do not edit -- it is v1's fingerprint, measured offline.
V1_BUNDLE_SHA256 = "99a2242e73cb91a64a39c9bb3fbfe9f2d34fa9bf3b42f6f62faae6524f70b956"

RESULTS_SUBDIR = "results/v2_kaggle"        # never touches results/v1_reference/
RUN_STEM       = "okf_rag_v2_kaggle"        # -> raw_okf_rag_v2_kaggle_full.csv / scored_...

# -- run-scope flags: flip in ORDER to advance past each safety stop --------
ALLOW_SMOKE      = True    # False => hard-stop right after the verification gate (bundle only)
ALLOW_FULL_RUN   = True    # <-- APPROVED 2026-09-02: run all 65 (smoke still runs first, separate files)
ACTIVE_QIDS_FILE = "results/v2_active_qids_v2.txt"   # the rebuilt 65 (was v2_active_qids.txt / 63)
SMOKE_QIDS       = ["Q001", "Q109", "Q081"]           # single / genuine-multi / unanswerable

import os, shutil, subprocess, sys
from pathlib import Path

def sh(cmd, **kw):
    """Run a shell command, stream its output, raise on a non-zero exit."""
    print(f"$ {cmd}", flush=True)
    return subprocess.run(cmd, shell=True, check=kw.pop("check", True), **kw)

print("GEN  :", GEN_MODEL)
print("JUDGE:", JUDGE_MODEL)

## 1 — GPU check

In [ ]:
sh("nvidia-smi", check=False)

import torch
print(f"\ntorch sees {torch.cuda.device_count()} GPU(s)")
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"  [{i}] {p.name}  {p.total_memory / 2**30:.1f} GiB  sm_{p.major}{p.minor}")

assert torch.cuda.device_count() >= 1, "No GPU attached -- set Accelerator to 'GPU T4 x2'."
if torch.cuda.device_count() < 2:
    print("\n[warn] expected 2 GPUs (T4 x2); a 30B model may not fit on one.")

## 2 — Project files (GitHub clone, else Kaggle dataset)

In [ ]:
proj = Path(PROJECT)

if (proj / "src" / "run_experiment.py").exists():
    print(f"SOURCE: existing tree at {proj} (session resume) -- not re-fetching.")
elif REPO_URL:
    sh(f"git clone --depth 1 {REPO_URL} {proj}")
    print(f"SOURCE: git clone from {REPO_URL}")
else:
    # Kaggle's mount path for an attached dataset is not stable: it has been seen at
    # /kaggle/input/<slug> AND at /kaggle/input/datasets/<owner>/<slug>/<slug>. A batch
    # run died on a hardcoded guess (2026-08-28), so find the tree by its contents.
    MARKER = Path("src") / "run_experiment.py"
    cands = [Path(KAGGLE_INPUT)] + sorted(
        p.parent.parent for p in Path("/kaggle/input").rglob(f"src/{MARKER.name}")
    )
    src = next((c for c in cands if (c / MARKER).exists()), None)
    assert src is not None, (
        f"No REPO_URL set and no project tree found under /kaggle/input "
        f"(looked for */{MARKER}). Tried: {[str(c) for c in cands]}. "
        f"Attach the Kaggle dataset described in the setup checklist (item 3)."
    )
    if src != Path(KAGGLE_INPUT):
        print(f"NOTE: KAGGLE_INPUT={KAGGLE_INPUT} was wrong; resolved to {src}")
    # Kaggle input is read-only and results must be writable, so copy the tree out.
    shutil.copytree(src, proj, dirs_exist_ok=True)
    print(f"SOURCE: copied Kaggle dataset {src} -> {proj}")

# Fail here, naming the missing path, rather than deep inside a retriever.
# data/okf_bundle is NOT required here: v2 rebuilds it fresh (cell "BUILD THE BUNDLE"),
# it must not already be in the dataset (that was v1's bundle -- see the 2026-08-28
# contamination audit). v2_active_qids.txt (63), not v2_qids.txt (v1's 60), is what
# this run actually uses -- but v2_qids.txt (v1's 60) is still needed: cell 21
# reconstructs the active set as (v2_qids - excluded + backfill) and asserts it matches.
REQUIRED = [
    "src/run_experiment.py", "src/evaluate.py", "src/build_bundle.py", "config.yaml",
    "questions/questions.csv", "data/raw",
    ACTIVE_QIDS_FILE,
    "results/v1_reference/scored_okf_rag_v1_full.csv",
]
missing = [p for p in REQUIRED if not (proj / p).exists()]
assert not missing, (
    f"Missing from the project tree: {missing}\n"
    f"If the dataset mounted one level deeper (e.g. {KAGGLE_INPUT}/okf-rag-v2), "
    f"point KAGGLE_INPUT at that inner directory and re-run."
)

n_raw = len(list((proj / "data" / "raw").glob("*.md")))
bundled = (proj / "data" / "okf_bundle").exists()
assert not bundled, (
    f"{proj}/data/okf_bundle already exists in the fetched dataset -- it should NOT. "
    f"v2 builds its own bundle; a pre-shipped one would silently reintroduce v1 "
    f"contamination. Remove it from the dataset and re-upload."
)
print(f"OK -- {n_raw} raw docs (expect 40), no pre-shipped bundle (correct -- v2 builds its own).")

os.chdir(proj)          # `python -m src.X` resolves the package from cwd; no install needed
sys.path.insert(0, str(proj))
sh("pip install -q sentence-transformers tqdm pyyaml pandas requests")


## 3 — Ollama + models

`OLLAMA_MAX_LOADED_MODELS=1` matters: generation and judging are sequential stages, and
letting Ollama hold a ~24.5 GB generator and the judge resident at once is how you OOM a T4.
`OLLAMA_CONTEXT_LENGTH` matters too — Ollama's default input context is small and the
OpenAI-compatible endpoint does not expose `num_ctx`, so the 1800-token retrieval context
plus the answer would be silently truncated at the default.

In [ ]:
import time
import requests

if shutil.which("ollama") is None:
    # The installer ships ollama-linux-amd64.tar.zst; Kaggle's image has no zstd, and the
    # 2026-08-27 session had to install it by hand mid-run. Do it up front.
    sh("apt-get update -qq && apt-get install -y zstd -qq", check=False)
    sh("curl -fsSL https://ollama.com/install.sh | sh")

os.environ["OLLAMA_MAX_LOADED_MODELS"] = "1"     # never hold gen + judge at once
os.environ["OLLAMA_CONTEXT_LENGTH"]    = "8192"   # generation needs ~1712 prompt + 4000 out (<6k);
#   judge needs ~3.9k in + 1500 out (<5.4k). 8192 covers both with margin. Was 24576 for the
#   bundle EXTRACTION stage -- now skipped (checkpoints shipped) -- and that oversized KV
#   reservation is the likely cause of the Ollama OOM ~34 calls into kernel v12. Smaller
#   num_ctx = smaller per-request compute buffers = more headroom against fragmentation on
#   the tight Q4_K_M-on-T4x2 fit.
os.environ["OLLAMA_KEEP_ALIVE"]        = "30m"

def ollama_up():
    try:
        requests.get("http://localhost:11434/api/tags", timeout=2).raise_for_status()
        return True
    except Exception:
        return False

# Idempotent: re-running this cell must not spawn a second server.
if ollama_up():
    print("ollama already serving.")
else:
    subprocess.Popen(["ollama", "serve"], stdout=open("/tmp/ollama.log", "wb"),
                     stderr=subprocess.STDOUT, env=os.environ.copy())
    for _ in range(60):
        if ollama_up():
            print("ollama up.")
            break
        time.sleep(2)
    else:
        raise SystemExit("ollama did not come up -- see /tmp/ollama.log")

def ollama_restart():
    """Kill and relaunch `ollama serve` -- used by sh_resumable when a batch dies
    (e.g. a mid-run VRAM OOM on the tight Q4_K_M/T4x2 fit)."""
    sh("kill -9 $(pgrep -f \"ollama\") 2>/dev/null || true", check=False)
    time.sleep(4)
    subprocess.Popen(["ollama", "serve"], stdout=open("/tmp/ollama.log", "ab"),
                     stderr=subprocess.STDOUT, env=os.environ.copy())
    for _ in range(90):
        if ollama_up():
            print("ollama restarted.")
            return
        time.sleep(2)
    raise SystemExit("ollama did not restart -- see /tmp/ollama.log")

def sh_resumable(cmd, tries=4):
    """Run cmd; on non-zero exit, restart ollama and retry. The pipeline is
    checkpointed (src/config.done_keys skips finished (arm,qid) rows), so each
    retry RESUMES rather than repeats. Lets a long unattended run survive an
    Ollama crash instead of losing the whole session."""
    for k in range(tries):
        try:
            return sh(cmd)
        except subprocess.CalledProcessError:
            if k == tries - 1:
                raise
            print(f"[resumable] attempt {k + 1}/{tries} failed -- restarting ollama and resuming")
            ollama_restart()

for m in (GEN_MODEL, JUDGE_MODEL):
    sh(f"ollama pull {m}")

sh("ollama list")

In [ ]:
# Load each model once and report the VRAM it actually takes, so "it fits" is measured
# rather than assumed -- a pull that succeeded can still fail to load.
def vram():
    return subprocess.run(
        "nvidia-smi --query-gpu=index,memory.used,memory.total --format=csv,noheader",
        shell=True, capture_output=True, text=True).stdout.strip()

for m in (GEN_MODEL, JUDGE_MODEL):
    print(f"\n--- loading {m} ---", flush=True)
    r = requests.post(
        "http://localhost:11434/v1/chat/completions",
        json={"model": m, "messages": [{"role": "user", "content": "Reply with: ok"}],
              "max_tokens": 2000, "temperature": 0.0},
        timeout=1800,
    )
    if r.status_code != 200:
        raise SystemExit(
            f"{m} failed to load: {r.status_code} {r.text[:400]}\n"
            f"If this is an out-of-memory error the model does not fit T4x2 -- "
            f"see the VRAM warning cell and pick another."
        )
    print("reply:", (r.json()["choices"][0]["message"]["content"] or "").strip()[:200])
    print("VRAM (idx, used, total):")
    print(vram())
    sh(f"ollama stop {m}", check=False)   # free it before loading the next

## 4 — v2 config (points the existing pipeline at local Ollama)

In [ ]:
# Generated from the variables above rather than checked in, so the config can never
# disagree with the models this session actually pulled. `extends: config.yaml` means
# retrieval / embedding / bundle path / questions / seed stay byte-identical to v1.
CFG = Path("config_v2_kaggle.yaml")
CFG.write_text(f"""# GENERATED BY notebooks/okf_rag_v2_kaggle.ipynb -- do not hand-edit.
extends: config.yaml

experiment:
  name: "{RUN_STEM}"

paths:
  results: "{RESULTS_SUBDIR}"

generation:
  model: "{GEN_MODEL}"
  base_url: "http://localhost:11434"
  api_key_env: null
  temperature: 0.0
  # NO reasoning_effort override -- kernel v15 set it to "none" and that was a
  # MISTAKE: the model collapsed into a keyword extractor that ignored the
  # SYSTEM_PROMPT format (1-3 sentences, SOURCES: line, exact "INSUFFICIENT
  # CONTEXT" for unanswerable) and even answered unanswerable Qs from outside
  # knowledge (Raft, GCP). v14 with reasoning ON produced correctly-formatted
  # answers -- only 6/130 rows spiraled on a runaway <think>. Keep reasoning ON.
  # 8000, not 4000: v14's 6 failures hit finish_reason=length at 4000; the 124
  # that completed peaked at 3132 completion tokens, so 8000 gives real headroom.
  # A genuine unbounded runaway still hits the cap -> empty -> LLMError -> the
  # per-row skip in run_experiment logs it to skipped_<stem>_<arm>.json and the
  # run continues.
  max_tokens: 8000
  timeout_s: 600
  retries: 5

bundle_builder:
  # v2 REBUILD: same model as generation, NOT v1's llama3.1:8b.
  model: "{BUNDLE_MODEL}"
  base_url: "http://localhost:11434"
  api_key_env: null
  temperature: 0.0
  send_seed: true
  # 16000 is v1's value, confirmed good for llama3.1:8b (zero truncations). It is
  # NOT enough for Nemotron on this task: the first rebuild attempt (2026-08-28)
  # burned the ENTIRE 16000-token budget on hidden reasoning and returned an
  # EMPTY completion on 8/8 retries for the first document (architecture.md,
  # 8.1KB -- not even the largest doc). That is a runaway chain-of-thought, not
  # a budget shortfall: raising max_tokens further would not have fixed it.
  # Fix: reasoning_effort "none" via Ollama's OpenAI-compatible endpoint
  # (documented top-level field: none/low/medium/high/max). Extraction is
  # structured information-extraction, not a task that needs deep reasoning --
  # Nemotron's own docs describe answering without reasoning traces as the
  # intended mode for exactly this kind of task. Generation and judge configs
  # are untouched; this key only exists on this block.
  reasoning_effort: "none"
  max_tokens: 16000
  timeout_s: 1800
  retries: 8
  max_retry_after_s: 120
  max_concepts_per_doc: 12
  min_concept_words: 15
  link_mode: "alias"

judge:
  enabled: true
  model: "{JUDGE_MODEL}"
  base_url: "http://localhost:11434"
  api_key_env: null
  temperature: 0.0
  max_tokens: 1500
  timeout_s: 600
  retries: 5
""", encoding="utf-8")

# make_judge() reads generation.api_key_env for the judge's credential; null there means
# "local provider, no auth header", which is what both roles want here.
print(CFG.read_text())

RESULTS = Path(RESULTS_SUBDIR)
RESULTS.mkdir(parents=True, exist_ok=True)
RAW    = RESULTS / f"raw_{RUN_STEM}_full.csv"
SCORED = RESULTS / f"scored_{RUN_STEM}_full.csv"
print("raw    ->", RAW)
print("scored ->", SCORED)

## 4b - BUILD THE BUNDLE (Nemotron)

v1's bundle was built by `llama3.1:8b`. The 2026-08-28 run reused it verbatim and was
therefore not a v2 result at all - it is archived as `results/v2_SUPERSEDED_V1BUNDLE/`.
This stage rebuilds the bundle from the same 40 raw documents using the **generation
model**, so Arm B runs on a genuinely v2 substrate.

This run **reuses the verified Nemotron bundle via shipped per-doc checkpoints**
(`results/v2_kaggle/bundle_debug/<doc>.concepts.json`, 40 files, from kernel v10). Extraction
is therefore SKIPPED - only deterministic cross-linking + `write_bundle` run (~1-2 min). If
the checkpoints are ever absent this cell falls back to a full ~30-60 min re-extraction.

**The next cell is a hard gate.** It refuses to let generation start unless the bundle
is provably new. Do not skip it, and do not "fix" it by relaxing an assert.

In [ ]:
# Build the v2 bundle. Deterministic cross-linking, LLM-assisted extraction.
BUNDLE_DIR = Path("data/okf_bundle")
DEBUG_DIR  = RESULTS / "bundle_debug"

# The per-doc checkpoint is keyed on FILENAME ONLY, so a checkpoint written by a
# different model is silently reused and would poison the rebuild. This is exactly
# the contamination the 2026-08-28 audit went looking for -- assert it cannot happen.
stale = sorted(DEBUG_DIR.glob("*.concepts.json")) if DEBUG_DIR.exists() else []
if stale:
    print(f"{len(stale)} per-doc checkpoints present in {DEBUG_DIR}.")
    print("   EXPECTED this run: the verified Nemotron checkpoints are shipped in the")
    print("   dataset so extraction is skipped. build_bundle re-checks each against data/raw/.")
    print(f"   If you ever need a clean re-extraction: shutil.rmtree('{DEBUG_DIR}')")
else:
    print(f"no checkpoints in {DEBUG_DIR} -- build_bundle will do a full ~30-60 min extraction")

# A leftover bundle would be partially overwritten rather than replaced -- start clean.
if BUNDLE_DIR.exists():
    print(f"!! {BUNDLE_DIR} already exists with "
          f"{sum(1 for _ in BUNDLE_DIR.rglob('*.md'))} concept files -- removing it first")
    shutil.rmtree(BUNDLE_DIR)

n_docs_in = sum(1 for _ in Path("data/raw").glob("*.md"))
assert n_docs_in == 40, f"expected 40 raw docs, found {n_docs_in} -- wrong dataset attached?"
print(f"extracting from {n_docs_in} raw docs with {BUNDLE_MODEL}")
print()

sh(f"python -m src.build_bundle --config {CFG}")

## 4c - VERIFICATION GATE - the bundle must be provably new

Three independent checks. **All three must pass before any generation runs.**

1. **Content hash** - SHA-256 over the sorted bundle tree must NOT equal v1's
   fingerprint. This is the check that would have caught the 2026-08-28 mistake.
2. **Model string** - the config that built it must name Nemotron, not `llama3.1:8b`.
3. **Timestamp** - every concept file must have been written during this session.

It also writes `bundle_provenance.json`, which records model + timestamp + tree hash.
`bundle_stats_*.json` records none of those, which is why v1's origin could only be
established by hashing after the fact.

In [ ]:
import hashlib, datetime, json as _json
NL = chr(10)

def tree_sha256(root: Path) -> str:
    """SHA-256 over (relative path, file content) for every file, sorted. Order-stable."""
    h = hashlib.sha256()
    for rel in sorted(q.relative_to(root).as_posix()
                      for q in root.rglob("*") if q.is_file()):
        h.update(rel.encode())
        h.update(hashlib.sha256((root / rel).read_bytes()).hexdigest().encode())
    return h.hexdigest()

files = [q for q in BUNDLE_DIR.rglob("*") if q.is_file()]
assert files, f"{BUNDLE_DIR} is empty -- the build cell did not produce a bundle"
new_sha = tree_sha256(BUNDLE_DIR)

# -- 1. content differs from v1 -------------------------------------------------
print(f"[1/3] tree sha256 : {new_sha}")
print(f"      v1          : {V1_BUNDLE_SHA256}")
assert new_sha != V1_BUNDLE_SHA256, (
    "STOP. The bundle is byte-identical to v1's. Generation would produce another "
    "v1 result. Delete data/okf_bundle/ and the bundle_debug checkpoints, then rebuild.")
print("      -> DIFFERENT from v1. PASS")
print()

# -- 2. built by the model we think ----------------------------------------------
cfg_txt = CFG.read_text(encoding="utf-8")
# the bundle_builder block runs until the next top-level key, which is judge:
bb = cfg_txt.split("bundle_builder:", 1)[1].split(NL + "judge:", 1)[0]
print(f"[2/3] bundle_builder.model -> {BUNDLE_MODEL}")
assert f'model: "{BUNDLE_MODEL}"' in bb, "config's bundle_builder.model is not BUNDLE_MODEL"
# ignore YAML comments: cell 13 legitimately writes "llama3.1:8b" into explanatory
# comments in this block ("...NOT v1's llama3.1:8b"), so a raw substring scan false-
# positives. Check only the ACTIVE model: directive(s).
bb_model_lines = [l.split("#", 1)[0].strip() for l in bb.splitlines()
                  if l.split("#", 1)[0].strip().startswith("model:")]
assert bb_model_lines == [f'model: "{BUNDLE_MODEL}"'], (
    f"config's bundle_builder has unexpected model: line(s): {bb_model_lines}")
assert not any("llama3.1" in l for l in bb_model_lines), (
    "STOP. bundle_builder is still pointed at v1's llama3.1:8b.")
print("      -> Nemotron, not llama3.1:8b. PASS")
print()

# -- 3. written during this session ----------------------------------------------
newest = max(q.stat().st_mtime for q in files)
oldest = min(q.stat().st_mtime for q in files)
age_h  = (time.time() - oldest) / 3600
fmt = lambda t: datetime.datetime.fromtimestamp(t).isoformat(timespec="seconds")
print(f"[3/3] {len(files)} files written {fmt(oldest)} .. {fmt(newest)}")
assert age_h < 12, (
    f"STOP. Oldest concept file is {age_h:.1f}h old -- older than this session. "
    "These are not freshly built files.")
print(f"      -> all written {age_h:.2f}h ago, within this session. PASS")
print()

stats_path = RESULTS / "bundle_stats_alias.json"
stats = _json.loads(stats_path.read_text(encoding="utf-8")) if stats_path.exists() else {}
prov = {
    "built_at_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
    "bundle_builder_model": BUNDLE_MODEL,
    "gen_quant": GEN_QUANT,
    "tree_sha256": new_sha,
    "v1_tree_sha256": V1_BUNDLE_SHA256,
    "differs_from_v1": new_sha != V1_BUNDLE_SHA256,
    "n_concept_files": len(files),
    "n_source_docs": n_docs_in,
    "concepts": stats.get("concepts"),
    "cross_links": stats.get("cross_links"),
    "link_mode": stats.get("link_mode"),
}
(RESULTS / "bundle_provenance.json").write_text(_json.dumps(prov, indent=2), encoding="utf-8")
print(_json.dumps(prov, indent=2))
print(f"v1 was 371 concepts / 1103 links -- a different count here is EXPECTED "
      f"(different extractor), not a bug.")
print("GATE PASSED. Generation may proceed.")

## STOP HERE (this run) - hard halt after the gate

This run is for bundle rebuild + verification ONLY, per instruction: do not proceed
into generation until the new multi-hop questions are written and the active qid
list is finalized. The next cell raises deliberately so a full "Run All" cannot
slide into generation unattended.

In [ ]:
if not ALLOW_SMOKE:
    raise SystemExit(
        "STOP: bundle rebuilt + verification gate passed. This is the safe halt.\n"
        "To run the 3-qid SMOKE TEST: set ALLOW_SMOKE=True in the settings cell and re-run.\n"
        "The full 65-qid run stays separately blocked by ALLOW_FULL_RUN."
    )
print("ALLOW_SMOKE=True -> proceeding to the 3-qid SMOKE TEST only "
      "(full 65-qid run is still blocked by ALLOW_FULL_RUN).")


## 5 — The 65 active qids (rebuilt 2026-09-02: 37 single / 18 verified-genuine multi / 10 unanswerable)

In [ ]:
import pandas as pd

# v2 ACTIVE SET, rebuilt 2026-09-02 -> results/v2_active_qids_v2.txt (65 qids).
# Uses ONLY verified-genuine two-document multi-hop questions; every degenerate
# "multi" from the old 63-qid list was dropped. Old list kept for reference.
# See results/v2_active_qids_v2.txt.README for the full derivation.
QIDS = Path(ACTIVE_QIDS_FILE).read_text(encoding="utf-8").split()
assert len(QIDS) == 65, f"{ACTIVE_QIDS_FILE} has {len(QIDS)} qids, expected 65"
assert len(set(QIDS)) == 65, "duplicate qid in the active list"

FLAWED = {"Q016", "Q048", "Q049", "Q058", "Q066", "Q076", "Q079"}
assert not (set(QIDS) & FLAWED), f"a known-flawed qid is in the active set: {sorted(set(QIDS) & FLAWED)}"

GENUINE_MULTI = {"Q056", "Q057", "Q064", "Q070", "Q072", "Q100", "Q102", "Q104",
                 "Q109", "Q111", "Q115", "Q126", "Q127", "Q128", "Q129", "Q132",
                 "Q133", "Q134"}

q = pd.read_csv("questions/questions.csv", keep_default_na=False)
assert set(QIDS) <= set(q["qid"]), "active list names a qid absent from questions.csv"
hop = q[q["qid"].isin(QIDS)]["hop_type"].value_counts().to_dict()
assert hop == {"single": 37, "multi": 18, "unanswerable": 10}, f"unexpected hop mix: {hop}"
# every 'multi' row in the active set must be a verified-genuine one
active_multi = {x for x in QIDS if q.set_index("qid").loc[x, "hop_type"] == "multi"}
assert active_multi == GENUINE_MULTI, f"active multi != genuine pool: {active_multi ^ GENUINE_MULTI}"

QIDS_FILE  = Path(ACTIVE_QIDS_FILE)
SMOKE_FILE = RESULTS / "v2_qids_smoke.txt"
RESULTS.mkdir(parents=True, exist_ok=True)
for x in SMOKE_QIDS:
    assert x in QIDS, f"smoke qid {x} is not in the active set"
SMOKE_FILE.write_text(chr(10).join(SMOKE_QIDS), encoding="utf-8")

print(f"{len(QIDS)} active qids  ({hop['single']} single / {hop['multi']} multi / "
      f"{hop['unanswerable']} unanswerable)")
print(" ".join(QIDS))
print(f"smoke subset: {SMOKE_QIDS}  "
      f"hop_types={[q.set_index('qid').loc[x, 'hop_type'] for x in SMOKE_QIDS]}")


# 🔥 SMOKE TEST — RUN THIS FIRST

3 qids × 2 arms = 6 generations, then 6 judge calls. Writes to its own `*_smoke.csv`
files, so it cannot contaminate the real run. Read the printed answers and judge labels
before committing a long session to the other 62.

In [ ]:
SMOKE_RAW    = RESULTS / f"raw_{RUN_STEM}_smoke.csv"
SMOKE_SCORED = RESULTS / f"scored_{RUN_STEM}_smoke.csv"

sh(f"python -m src.run_experiment --config {CFG} --qids-file {SMOKE_FILE} "
   f"--arms chunk okf --out {SMOKE_RAW.name}")
sh(f"python -m src.evaluate --config {CFG} --input {SMOKE_RAW.name} --sample-for-human 0")

smoke = pd.read_csv(SMOKE_SCORED)
for _, r in smoke.iterrows():
    print("=" * 78)
    print(f"[{r['arm']}] {r['qid']}  {r['question']}")
    print(f"  gold  : {r['gold_answer']}")
    print(f"  answer: {r['answer']}")
    print(f"  judge : {r['judge_label']}   em={r['em']} f1={r['f1']:.3f} "
          f"recall={r['retrieval_recall']} halluc={r['hallucination']}")
    print(f"  gen   : finish={r.get('finish_reason')} "
          f"prompt_tok={r.get('prompt_tokens')} completion_tok={r.get('completion_tokens')} "
          f"n_expanded={r.get('n_expanded')} gen_s={r.get('generate_s')}")
print("=" * 78)
print(f"\n{len(smoke)} rows (expect 6). Happy with these? Continue below.")

In [ ]:
# ---- HARD STOP after the smoke test (intentional, reversible) ----
if not ALLOW_FULL_RUN:
    raise SystemExit(
        "SMOKE TEST COMPLETE -- 6 rows printed above. Review the answers and judge "
        "labels. To run the full 65 qids: set ALLOW_FULL_RUN=True in the settings cell "
        "(keep ALLOW_SMOKE=True) and re-run. This stop is deliberate."
    )
print("ALLOW_FULL_RUN=True -> proceeding to the full 65-qid run.")


# FULL RUN

Cells 6–8 are all **checkpointed and safely re-runnable**. `src/config.py`'s `done_keys()`
reads the output CSV and skips `(arm, qid)` pairs already present, and `append_row()`
writes one row at a time with the file closed after each — so a session killed at question
41 keeps 40 rows and resumes at 41. Just re-run the cell.

Delete the output CSV only if you want a genuinely clean rerun (e.g. after changing a
model), since a resumed file would otherwise mix rows produced under two configs.

## 6 — Generation, chunk arm (Arm A, baseline)

In [ ]:
sh_resumable(f"python -m src.run_experiment --config {CFG} --qids-file {QIDS_FILE} "
   f"--arms chunk --out {RAW.name}")

print(pd.read_csv(RAW).groupby("arm")["qid"].count())

## 7 — Generation, OKF arm (Arm B)

In [ ]:
sh_resumable(f"python -m src.run_experiment --config {CFG} --qids-file {QIDS_FILE} "
   f"--arms okf --out {RAW.name}")

print(pd.read_csv(RAW).groupby("arm")["qid"].count())

## 8 — Judging (all 130 rows)

Uses `src/evaluate.py` unchanged, so `JUDGE_SYSTEM` / `JUDGE_USER` — the rubric — are
byte-identical to v1's. That is what makes v1 and v2 judge labels comparable; do not
reword the prompt here.

`--sample-for-human 126` also writes a labelling sheet, so v2 can eventually get human
labels the same way v1 did.

In [ ]:
sh_resumable(f"python -m src.evaluate --config {CFG} --input {RAW.name} --sample-for-human 130")

s = pd.read_csv(SCORED)
print(s.groupby("arm")[["em", "f1", "retrieval_recall", "hallucination"]].mean())

## 9 — ⚠️ SAVE AND EXPORT — run this before the session ends

**Everything outside `/kaggle/working/` is destroyed when this session ends or
disconnects.** After running this cell, open the notebook's **Data → Output** panel and
**download `okf_rag_v2_kaggle_results.zip`**. Nothing else will survive.

(The project was cloned into `/kaggle/working/okf-rag`, so the CSVs are already inside the
persistent directory — the zip is one convenient download and insurance against the
working tree being cleaned.)

In [ ]:
import zipfile

ZIP = Path("/kaggle/working/okf_rag_v2_kaggle_results.zip")
with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for f in sorted(RESULTS.rglob("*")):
        if f.is_file():
            z.write(f, f.relative_to(RESULTS.parent))
    z.write(CFG, CFG.name)

print(f"{ZIP}  ({ZIP.stat().st_size / 1024:.0f} KB)")
for n in zipfile.ZipFile(ZIP).namelist():
    print("  ", n)
print("\n>>> DOWNLOAD THIS FILE FROM THE OUTPUT PANEL BEFORE THE SESSION ENDS. <<<")

## 10 — Final summary

In [ ]:
raw    = pd.read_csv(RAW)
scored = pd.read_csv(SCORED)

print(f"raw rows   : {len(raw)}   (expect 130)")
print(f"scored rows: {len(scored)} (expect 130)")
print(raw.groupby("arm")["qid"].nunique().rename("qids per arm"))

expected = set(QIDS)
ok = True
for name, df in (("generation", raw), ("judging", scored)):
    for arm in ("chunk", "okf"):
        got = set(df[df["arm"] == arm]["qid"].astype(str))
        if got != expected:
            ok = False
            print(f"[MISSING] {name}/{arm}: {sorted(expected - got)}")
            if got - expected:
                print(f"[EXTRA]   {name}/{arm}: {sorted(got - expected)}")

bad = scored[~scored["judge_label"].isin(
    ["supported", "unsupported", "contradicted", "abstained"])]
if len(bad):
    ok = False
    print(f"[WARN] {len(bad)} unparsed judge labels: "
          f"{bad[['arm', 'qid', 'judge_label']].to_dict('records')}")

print("\nCOMPLETE -- all 65 qids present in both arms, both stages." if ok
      else "\nINCOMPLETE -- re-run the generation/judging cells; done_keys resumes from where they stopped.")